# 03 Extensive Description Rebuild

This notebook rebuilds the text layer used by BERTopic and the recommender.

Goals:

1. keep the original cleaned dataset intact
2. audit short, generic, and suspicious descriptions
3. fetch source-grounded long plot/premise text using IMDb `tconst -> Wikidata P345 -> English Wikipedia`
4. fix title-description mismatch risk by trusting identifier-matched source text over title-name matching
5. create a richer `modeling_description` and `model_text` for NLP clustering
6. write a new enriched dataset for the BERTopic notebook

Important accuracy rule: this notebook does not silently invent plot facts. If a long source-grounded plot cannot be found, the row is flagged for manual research instead of being filled with hallucinated detail.

## 1. Imports And Paths

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Iterable
from urllib.parse import quote, unquote, urlparse
import re
import time

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from IPython.display import display

pd.set_option("display.max_colwidth", 240)
pd.set_option("display.max_columns", 60)

DATA_DIR = Path("data")
BASE_DATA_PATH = DATA_DIR / "cleaned_watch_data.csv"

SOURCE_MAPPING_PATH = DATA_DIR / "extensive_wikidata_wikipedia_sources.csv"
LONG_PLOTS_PATH = DATA_DIR / "extensive_long_plot_summaries.csv"
REVIEW_PATH = DATA_DIR / "extensive_description_rebuild_review.csv"
AUDIT_PATH = DATA_DIR / "extensive_description_audit.csv"
FINAL_ENRICHED_PATH = DATA_DIR / "cleaned_watch_data_extensive_enriched.csv"
BERTOPIC_COMPAT_PATH = DATA_DIR / "cleaned_watch_data_phase2_enriched.csv"

USER_AGENT = "MovieMoodRecommenderStudentProject/2.0 (local source-grounded description enrichment)"
WIKIDATA_SPARQL_URL = "https://query.wikidata.org/sparql"
WIKIPEDIA_API_URL = "https://en.wikipedia.org/w/api.php"
WIKIPEDIA_REST_SUMMARY_URL = "https://en.wikipedia.org/api/rest_v1/page/summary/{title}"
REQUEST_TIMEOUT_SECONDS = 18.0

PREFERRED_SECTION_NAMES = {
    "plot", "plot summary", "premise", "synopsis", "story", "storyline", "overview"
}

REJECT_SECTION_NAMES = {
    "cast", "characters", "production", "release", "reception", "awards", "references",
    "external links", "notes", "see also", "soundtrack", "home media", "box office"
}

MIN_USEFUL_SOURCE_WORDS = 60
MIN_GOOD_MODELING_WORDS = 140
MAX_SOURCE_WORDS_FOR_MODELING = 900
REQUEST_PAUSE_SECONDS = 0.35

# Only use manual overrides after verifying that the page is the correct title/version.
# These handle cases where Wikidata has the IMDb ID but no English Wikipedia sitelink.
MANUAL_WIKIPEDIA_TITLE_OVERRIDES = {
    "tt8788458": {
        "wikidata_id": "Q61786454",
        "wikidata_label": "The Promised Neverland",
        "enwiki_url": "https://en.wikipedia.org/wiki/The_Promised_Neverland",
        "wikipedia_title": "The Promised Neverland",
        "override_reason": "manual_verified_wikipedia_page_for_matching_anime_franchise",
    },
}


## 2. Load And Audit Current Dataset

This section measures the real problem before rebuilding. Short descriptions and suspicious title/description patterns are the rows most likely to damage BERTopic.

In [ ]:
def word_count(text):
    return len(re.findall(r"\b\w+\b", str(text)))


def clean_text(text):
    if pd.isna(text):
        return ""
    text = re.sub(r"\[[^\]]*\]", " ", str(text))
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def truncate_words(text, max_words):
    words = str(text).split()
    return " ".join(words[:max_words]) if len(words) > max_words else str(text)


watch_df = pd.read_csv(BASE_DATA_PATH).reset_index(drop=True)
watch_df["row_id"] = np.arange(len(watch_df))
watch_df["description_word_count"] = watch_df["description"].fillna("").map(word_count)

print("Rows:", len(watch_df))
print("Missing descriptions:", int(watch_df["description"].isna().sum()))
print("Description word-count quantiles:")
display(watch_df["description_word_count"].quantile([0, .1, .25, .5, .75, .9, .95, .99, 1]).to_frame("words"))

audit = watch_df.copy()
audit["audit_priority"] = 0
audit.loc[audit["description_word_count"] < 20, "audit_priority"] += 5
audit.loc[audit["description_word_count"].between(20, 39), "audit_priority"] += 4
audit.loc[audit["description_word_count"].between(40, 69), "audit_priority"] += 3
audit.loc[audit["description_word_count"].between(70, 119), "audit_priority"] += 2
audit.loc[audit["description"].fillna("").str.contains("A film by|A story of|A tale of|A man |A woman ", case=False, regex=True), "audit_priority"] += 1

audit_columns = [
    "tconst", "content_type", "primary_title", "release_year", "genres", "average_rating", "num_votes",
    "description_word_count", "audit_priority", "description"
]
audit.sort_values(["audit_priority", "description_word_count", "num_votes"], ascending=[False, True, False])[audit_columns].to_csv(AUDIT_PATH, index=False)
print("Saved audit:", AUDIT_PATH)
display(audit.sort_values(["audit_priority", "description_word_count", "num_votes"], ascending=[False, True, False])[audit_columns].head(25))


## 3. Source-Grounded Wikipedia/Wikidata Helpers

The key safety decision is to map by IMDb ID, not by title search. This avoids grabbing the wrong `The Bad Guys`, `Invincible`, or same-title remake.

In [ ]:
@dataclass
class EnrichedSummary:
    tconst: str
    primary_title: str
    release_year: int | str
    wikidata_id: str
    wikidata_label: str
    wikipedia_title: str
    wikipedia_url: str
    source_section: str
    enrichment_text: str
    enrichment_word_count: int
    current_description: str
    current_description_word_count: int
    match_method: str
    license_note: str


def batched(values: list[str], batch_size: int) -> Iterable[list[str]]:
    for start in range(0, len(values), batch_size):
        yield values[start:start + batch_size]


def wikipedia_title_from_url(url: str) -> str:
    path = urlparse(str(url)).path
    title = path.rsplit("/", 1)[-1]
    return unquote(title).replace("_", " ")


def request_json(session, url, *, params=None, retries=3, pause_seconds=1.2, timeout_seconds=None):
    timeout_seconds = REQUEST_TIMEOUT_SECONDS if timeout_seconds is None else timeout_seconds
    last_error = None
    for attempt in range(retries):
        try:
            response = session.get(url, params=params, timeout=timeout_seconds)
            if response.status_code in {429, 500, 502, 503, 504}:
                time.sleep(pause_seconds * (attempt + 2))
                continue
            response.raise_for_status()
            return response.json()
        except Exception as exc:
            last_error = exc
            time.sleep(pause_seconds * (attempt + 1))
    raise RuntimeError(f"Request failed after retries: {url}") from last_error


def fetch_wikidata_mappings(session, tconsts, *, batch_size=80, pause_seconds=0.35):
    rows = []
    for batch_number, batch in enumerate(batched(list(tconsts), batch_size), start=1):
        values = " ".join(f'"{item}"' for item in batch)
        query = f'''
SELECT ?imdb ?item ?itemLabel ?enwiki WHERE {{
  VALUES ?imdb {{ {values} }}
  ?item wdt:P345 ?imdb .
  OPTIONAL {{
    ?enwiki schema:about ?item ;
            schema:isPartOf <https://en.wikipedia.org/> .
  }}
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
}}
'''
        payload = request_json(session, WIKIDATA_SPARQL_URL, params={"query": query, "format": "json"})
        for binding in payload["results"]["bindings"]:
            item_url = binding["item"]["value"]
            rows.append({
                "tconst": binding["imdb"]["value"],
                "wikidata_id": item_url.rsplit("/", 1)[-1],
                "wikidata_url": item_url,
                "wikidata_label": binding.get("itemLabel", {}).get("value", ""),
                "enwiki_url": binding.get("enwiki", {}).get("value", ""),
            })
        print(f"Wikidata batch {batch_number}: {len(rows)} mapped rows so far", flush=True)
        time.sleep(pause_seconds)

    mappings = pd.DataFrame(rows).drop_duplicates() if rows else pd.DataFrame(columns=["tconst", "wikidata_id", "wikidata_url", "wikidata_label", "enwiki_url"])
    mappings["wikipedia_title"] = mappings["enwiki_url"].map(lambda url: wikipedia_title_from_url(url) if isinstance(url, str) and url else "")
    return mappings


def apply_manual_source_overrides(mappings):
    mappings = mappings.copy()
    for tconst, override in MANUAL_WIKIPEDIA_TITLE_OVERRIDES.items():
        mask = mappings["tconst"].astype(str).eq(tconst)
        if not mask.any():
            continue
        for column, value in override.items():
            mappings.loc[mask, column] = value
    return mappings


def fetch_page_sections(session, title):
    payload = request_json(session, WIKIPEDIA_API_URL, params={
        "action": "parse", "page": title, "prop": "sections", "format": "json", "redirects": "1"
    })
    return payload.get("parse", {}).get("sections", [])


def parse_section_html(session, title, section_index):
    payload = request_json(session, WIKIPEDIA_API_URL, params={
        "action": "parse", "page": title, "section": section_index, "prop": "text", "format": "json",
        "redirects": "1", "disableeditsection": "1"
    })
    html = payload.get("parse", {}).get("text", {}).get("*", "")
    soup = BeautifulSoup(html, "html.parser")
    for unwanted in soup.select("sup, table, .mw-editsection, .reference, .noprint, style, script, figure"):
        unwanted.decompose()
    paragraphs = []
    for paragraph in soup.find_all("p"):
        text = clean_text(paragraph.get_text(" ", strip=True))
        if text:
            paragraphs.append(text)
    return clean_text(" ".join(paragraphs))


def fetch_rest_summary(session, title):
    payload = request_json(session, WIKIPEDIA_REST_SUMMARY_URL.format(title=quote(title, safe="")))
    return clean_text(payload.get("extract", ""))


def choose_plot_section(sections):
    for section in sections:
        line = clean_text(section.get("line", "")).lower()
        if line in PREFERRED_SECTION_NAMES:
            return section
    for section in sections:
        line = clean_text(section.get("line", "")).lower()
        if any(name in line for name in PREFERRED_SECTION_NAMES) and line not in REJECT_SECTION_NAMES:
            return section
    return None


## 4. Run Extensive Enrichment

By default this targets **all rows**, not only weak descriptions. It checkpoints repeatedly, so it can be stopped and resumed.

For a small test first, call `run_extensive_enrichment(limit=50)`. For the real rebuild, call `run_extensive_enrichment(limit=None)`.

In [ ]:
def enrich_row(session, source_row, mapping_row, *, min_words=MIN_USEFUL_SOURCE_WORDS, pause_seconds=REQUEST_PAUSE_SECONDS):
    title = mapping_row.get("wikipedia_title", "")
    if not isinstance(title, str) or not title.strip():
        return None, {
            "tconst": source_row["tconst"],
            "primary_title": source_row["primary_title"],
            "release_year": source_row["release_year"],
            "reason": "no_english_wikipedia_page_matched_by_tconst",
            "current_description": source_row["description"],
        }

    sections = fetch_page_sections(session, title)
    section = choose_plot_section(sections)
    source_section = ""
    enrichment_text = ""
    match_method = "wikidata_p345_to_enwiki"

    if section is not None:
        source_section = clean_text(section.get("line", ""))
        enrichment_text = parse_section_html(session, title, str(section["index"]))
        time.sleep(pause_seconds)

    if word_count(enrichment_text) < min_words:
        fallback_text = fetch_rest_summary(session, title)
        time.sleep(pause_seconds)
        if word_count(fallback_text) > word_count(enrichment_text):
            enrichment_text = fallback_text
            source_section = "REST summary fallback"
            match_method = "wikidata_p345_to_enwiki_rest_summary_fallback"

    wc = word_count(enrichment_text)
    if wc < min_words:
        return None, {
            "tconst": source_row["tconst"],
            "primary_title": source_row["primary_title"],
            "release_year": source_row["release_year"],
            "wikidata_id": mapping_row.get("wikidata_id", ""),
            "wikidata_label": mapping_row.get("wikidata_label", ""),
            "wikipedia_title": title,
            "wikipedia_url": mapping_row.get("enwiki_url", ""),
            "reason": f"source_text_too_short:{wc}_words",
            "current_description": source_row["description"],
        }

    return EnrichedSummary(
        tconst=source_row["tconst"],
        primary_title=source_row["primary_title"],
        release_year=source_row["release_year"],
        wikidata_id=mapping_row.get("wikidata_id", ""),
        wikidata_label=mapping_row.get("wikidata_label", ""),
        wikipedia_title=title,
        wikipedia_url=mapping_row.get("enwiki_url", ""),
        source_section=source_section,
        enrichment_text=enrichment_text,
        enrichment_word_count=wc,
        current_description=source_row["description"],
        current_description_word_count=word_count(source_row["description"]),
        match_method=match_method,
        license_note="Source-grounded text extracted from English Wikipedia page linked to matching Wikidata IMDb ID (P345). Review attribution requirements before redistribution.",
    ), None


def run_extensive_enrichment(
    input_path=BASE_DATA_PATH,
    output_path=LONG_PLOTS_PATH,
    sources_output_path=SOURCE_MAPPING_PATH,
    review_output_path=REVIEW_PATH,
    limit=None,
    min_enrichment_words=MIN_USEFUL_SOURCE_WORDS,
    checkpoint_every=50,
    reuse_sources=True,
    resume=True,
    retry_fetch_errors=True,
    target_tconsts=None,
):
    df = pd.read_csv(input_path).reset_index(drop=True)
    if target_tconsts is not None:
        target_tconsts = set(str(value) for value in target_tconsts)
        df = df[df["tconst"].astype(str).isin(target_tconsts)].copy()
    if limit is not None:
        df = df.head(limit).copy()
    print(f"Loaded {len(df)} rows for extensive enrichment")

    session = requests.Session()
    session.headers.update({"User-Agent": USER_AGENT})

    source_path = Path(sources_output_path)
    id_columns = ["tconst", "wikidata_id", "wikidata_url", "wikidata_label", "enwiki_url", "wikipedia_title"]
    if reuse_sources and source_path.exists() and source_path.stat().st_size > 0:
        try:
            source_map = pd.read_csv(source_path)
        except pd.errors.EmptyDataError:
            source_map = pd.DataFrame(columns=id_columns)

        known_tconsts = set(source_map.get("tconst", pd.Series(dtype=str)).dropna().astype(str))
        missing_tconsts = [value for value in df["tconst"].dropna().astype(str).tolist() if value not in known_tconsts]
        if missing_tconsts:
            print(f"Source mapping is partial; fetching {len(missing_tconsts)} missing mappings")
            missing_source_map = fetch_wikidata_mappings(session, missing_tconsts)
            source_map = pd.concat([source_map, missing_source_map], ignore_index=True).drop_duplicates("tconst", keep="last")
            source_map.to_csv(sources_output_path, index=False)
            print(f"Updated source mapping: {sources_output_path}")
        else:
            print(f"Reused complete source mapping: {sources_output_path}")

        mappings = df[["tconst", "primary_title", "original_title", "display_title", "release_year", "genres", "description"]].merge(
            source_map[[column for column in id_columns if column in source_map.columns]].drop_duplicates("tconst"),
            on="tconst",
            how="left",
        )
    else:
        source_map = fetch_wikidata_mappings(session, df["tconst"].dropna().astype(str).tolist())
        mappings = source_map.merge(
            df[["tconst", "primary_title", "original_title", "display_title", "release_year", "genres", "description"]],
            on="tconst",
            how="right",
        )
        mappings.to_csv(sources_output_path, index=False)
        print(f"Wrote source mapping: {sources_output_path}")

    mappings = apply_manual_source_overrides(mappings)
    mappings["has_enwiki"] = mappings["enwiki_url"].fillna("").ne("")
    print("Rows with English Wikipedia page via P345 or manual override:", int(mappings["has_enwiki"].sum()))

    enriched_rows = []
    review_rows = []
    completed_tconsts = set()

    output_file = Path(output_path)
    review_file = Path(review_output_path)

    if resume and output_file.exists():
        existing = pd.read_csv(output_file)
        if not existing.empty:
            enriched_rows = [EnrichedSummary(**row) for row in existing.to_dict(orient="records")]
            completed_tconsts.update(existing["tconst"].astype(str))
        print(f"Resuming with {len(enriched_rows)} existing enriched rows")

    if resume and review_file.exists() and review_file.stat().st_size > 0:
        try:
            existing_review = pd.read_csv(review_file)
        except pd.errors.EmptyDataError:
            existing_review = pd.DataFrame(columns=["tconst", "reason"])
        if not existing_review.empty:
            if retry_fetch_errors and "reason" in existing_review.columns:
                existing_review = existing_review[~existing_review["reason"].fillna("").astype(str).str.startswith("fetch_error:")].copy()
            review_rows = existing_review.to_dict(orient="records")
            completed_tconsts.update(existing_review["tconst"].astype(str))
        print(f"Resuming with {len(review_rows)} existing review rows")
    elif resume and review_file.exists():
        print("Review file exists but is empty; starting review rows from 0")

    source_by_tconst = df.set_index("tconst", drop=False)
    mapped = mappings.copy()

    for number, (_, mapping_row) in enumerate(mapped.iterrows(), start=1):
        tconst = mapping_row["tconst"]
        if str(tconst) in completed_tconsts:
            continue
        source_row = source_by_tconst.loc[tconst]
        try:
            enriched, review = enrich_row(session, source_row, mapping_row, min_words=min_enrichment_words)
            if enriched is not None:
                enriched_rows.append(enriched)
            if review is not None:
                review_rows.append(review)
            completed_tconsts.add(str(tconst))
        except Exception as exc:
            review_rows.append({
                "tconst": source_row["tconst"],
                "primary_title": source_row["primary_title"],
                "release_year": source_row["release_year"],
                "wikidata_id": mapping_row.get("wikidata_id", ""),
                "wikidata_label": mapping_row.get("wikidata_label", ""),
                "wikipedia_title": mapping_row.get("wikipedia_title", ""),
                "wikipedia_url": mapping_row.get("enwiki_url", ""),
                "reason": f"fetch_error:{type(exc).__name__}:{exc}",
                "current_description": source_row["description"],
            })
            completed_tconsts.add(str(tconst))

        if number % checkpoint_every == 0:
            pd.DataFrame([row.__dict__ for row in enriched_rows]).to_csv(output_path, index=False)
            pd.DataFrame(review_rows).to_csv(review_output_path, index=False)
            print(f"Checked {number}/{len(mapped)}; enriched {len(enriched_rows)}; review {len(review_rows)}")

    pd.DataFrame([row.__dict__ for row in enriched_rows]).to_csv(output_path, index=False)
    pd.DataFrame(review_rows).to_csv(review_output_path, index=False)
    print(f"Wrote enrichments: {output_path} ({len(enriched_rows)} rows)")
    print(f"Wrote review file: {review_output_path} ({len(review_rows)} rows)")
    return pd.DataFrame([row.__dict__ for row in enriched_rows]), pd.DataFrame(review_rows)


In [ ]:
# Smoke test first. Review the output, then run the full rebuild below.
# enriched_smoke_df, review_smoke_df = run_extensive_enrichment(
#     limit=50,
#     output_path=DATA_DIR / "extensive_smoke_long_plot_summaries.csv",
#     review_output_path=DATA_DIR / "extensive_smoke_description_rebuild_review.csv",
#     reuse_sources=True,
#     resume=True,
# )

# Full rebuild. This can take a while, but it checkpoints and resumes.
# enriched_df, review_df = run_extensive_enrichment(limit=None, reuse_sources=True, resume=True)


## 5. Build The New Enriched Dataset

This creates the file the BERTopic notebook already expects: `data/cleaned_watch_data_phase2_enriched.csv`.

`description` remains the original project description. `enrichment_text` stores the source-grounded plot/premise text. `modeling_description` is the richer text field the next notebook should use.

In [ ]:
def build_modeling_description(row):
    original = clean_text(row.get("description", ""))
    enrichment = clean_text(row.get("enrichment_text", ""))
    parts = []
    if original:
        parts.append(f"Official/current description: {original}")
    if enrichment:
        parts.append(f"Verified plot or premise: {truncate_words(enrichment, MAX_SOURCE_WORDS_FOR_MODELING)}")
    return " ".join(parts).strip()


def build_extensive_enriched_dataset(
    base_path=BASE_DATA_PATH,
    long_plots_path=LONG_PLOTS_PATH,
    review_path=REVIEW_PATH,
    final_path=FINAL_ENRICHED_PATH,
    bertopic_compat_path=BERTOPIC_COMPAT_PATH,
):
    base = pd.read_csv(base_path).reset_index(drop=True)
    base["row_id"] = np.arange(len(base))
    base["description_word_count"] = base["description"].fillna("").map(word_count)

    if Path(long_plots_path).exists():
        enriched = pd.read_csv(long_plots_path)
    else:
        enriched = pd.DataFrame(columns=["tconst", "enrichment_text", "enrichment_word_count", "wikipedia_url", "source_section", "match_method"])

    keep_cols = [
        "tconst", "wikidata_id", "wikidata_label", "wikipedia_title", "wikipedia_url", "source_section",
        "enrichment_text", "enrichment_word_count", "match_method", "license_note"
    ]
    keep_cols = [c for c in keep_cols if c in enriched.columns]
    enriched = enriched[keep_cols].drop_duplicates("tconst") if not enriched.empty else enriched

    output = base.merge(enriched, on="tconst", how="left")
    output["has_phase2_enrichment"] = output["enrichment_text"].fillna("").astype(str).str.strip().ne("")
    output["enrichment_word_count"] = output["enrichment_word_count"].where(output["has_phase2_enrichment"], np.nan)
    output["modeling_description"] = output.apply(build_modeling_description, axis=1)
    output["modeling_description_word_count"] = output["modeling_description"].map(word_count)
    output["needs_manual_research"] = output["modeling_description_word_count"] < MIN_GOOD_MODELING_WORDS
    output["description_rebuild_status"] = np.where(
        output["has_phase2_enrichment"],
        "source_grounded_long_text_added",
        "needs_manual_research_no_verified_long_source",
    )

    if Path(review_path).exists() and Path(review_path).stat().st_size > 0:
        try:
            review = pd.read_csv(review_path)
        except pd.errors.EmptyDataError:
            review = pd.DataFrame(columns=["tconst", "reason"])
        review_cols = [c for c in ["tconst", "reason"] if c in review.columns]
        if review_cols and not review.empty:
            output = output.merge(review[review_cols].drop_duplicates("tconst").rename(columns={"reason": "enrichment_review_reason"}), on="tconst", how="left")
        else:
            output["enrichment_review_reason"] = ""
    else:
        output["enrichment_review_reason"] = ""

    output.to_csv(final_path, index=False)
    output.to_csv(bertopic_compat_path, index=False)

    print("Saved extensive enriched dataset:", final_path)
    print("Saved BERTopic-compatible enriched dataset:", bertopic_compat_path)
    print("Rows:", len(output))
    print("Rows with source-grounded long text:", int(output["has_phase2_enrichment"].sum()))
    print("Rows still needing manual research:", int(output["needs_manual_research"].sum()))
    display(output[[
        "tconst", "primary_title", "release_year", "genres", "description_word_count", "has_phase2_enrichment",
        "enrichment_word_count", "modeling_description_word_count", "needs_manual_research", "description_rebuild_status"
    ]].head(20))
    return output


# Run this after the enrichment step has produced/updated LONG_PLOTS_PATH.
# rebuilt_df = build_extensive_enriched_dataset()


## 6. Check Known Problem Titles

Use this after building the enriched dataset to verify examples that previously caused bad recommendations.

In [ ]:
def inspect_titles(df, titles):
    mask = df["primary_title"].str.lower().isin([title.lower() for title in titles])
    cols = [
        "tconst", "content_type", "primary_title", "release_year", "genres",
        "description_word_count", "has_phase2_enrichment", "enrichment_word_count",
        "modeling_description_word_count", "needs_manual_research", "description", "enrichment_text"
    ]
    cols = [c for c in cols if c in df.columns]
    display(df.loc[mask, cols].sort_values(["primary_title", "release_year"]))


# if "rebuilt_df" in globals():
#     inspect_titles(rebuilt_df, ["Inside Out", "Invincible", "The Bad Guys"])


## 7. Next Step

After `cleaned_watch_data_phase2_enriched.csv` is rebuilt, rerun `01_bertopic_nlp_pipeline.ipynb` from the top, but update it to prefer `modeling_description` when present. Then delete stale embeddings/model outputs before rerunning, because old embeddings were built from the old weak text.